# Hyperparameter Tuning with Ray Tune

This notebook demonstrates how to tune a TSUT `Pipeline` with the `RayPipelineTuner`.

The tuner:

1. **Snapshots** the pipeline's config so every trial rebuilds an identical pipeline from scratch.
2. **Discovers** tunable hyperparameters from every node exposing a `hyperparameter_space` class attribute.
3. **Runs** Ray Tune trials, each of which re-applies the sampled hyperparameters, trains the pipeline, evaluates its metrics, and reports a single scalar objective.
4. **Reports** the best result via `get_best()`.

Along the way we will:

- Inspect the default hyperparameter space.
- Customise the search space.
- Pick the objective via either `optimization_metric` (a single metric name) or `metric_aggregator` (a callable that reduces all metrics to a scalar).

## 1. Build a Pipeline with Tunable Nodes

We reuse the CSV-based pipeline from `pipeline.ipynb`. Two of its nodes expose a `hyperparameter_space` and will automatically become tunable:

| Node | Hyperparameters |
| --- | --- |
| `MissingRateFilter` | `threshold` |
| `RandomForestRegressor` | `n_estimators`, `max_depth` |

In [ ]:
import sys
from pathlib import Path

from tsut import NODE_REGISTRY
from tsut.core.common.logging import configure
from tsut.core.pipeline.pipeline import Edge, Pipeline, PipelineConfig

configure(level="WARNING", stream=sys.stdout, fmt="text")

# Resolve data paths to absolute form so Ray workers (which may run from a
# different working directory) can still locate the CSVs.
data_dir = (Path.cwd() / ".." / "data").resolve()

# --- Data sources ---
csv_config_class = NODE_REGISTRY.get_node_config_class("TabularCSVFetcher")

data_csv_config = csv_config_class()
data_csv_config.running_config.csv_path = str(data_dir / "fake_batch.csv")
data_csv_config.running_config.context_path = str(data_dir / "fake_batch_context.json")

target_csv_config = csv_config_class()
target_csv_config.running_config.csv_path = str(data_dir / "fake_target_df.csv")
target_csv_config.running_config.context_path = str(data_dir / "fake_target_context.json")

# --- Transform configs ---
missing_filter_config_class = NODE_REGISTRY.get_node_config_class("MissingRateFilter")
data_missing_filter_config = missing_filter_config_class()
target_missing_filter_config = missing_filter_config_class()
target_missing_filter_config.in_ports["input"].mode = ["training", "evaluation"]
target_missing_filter_config.hyperparameters.threshold = 0.0

nodes = {
    "data_source": ("TabularCSVFetcher", data_csv_config),
    "target_source": ("TabularCSVFetcher", target_csv_config),
    "data_missing_filter": ("MissingRateFilter", data_missing_filter_config),
    "category_splitter": ("DataCategoryFilter", NODE_REGISTRY.get_node_config_class("DataCategoryFilter")()),
    "one_hot_encoder": ("OneHotEncoding", NODE_REGISTRY.get_node_config_class("OneHotEncoding")()),
    "target_missing_filter": ("MissingRateFilter", target_missing_filter_config),
    "iqr_filter": ("IQROutlierFilter", NODE_REGISTRY.get_node_config_class("IQROutlierFilter")()),
    "variance_threshold_filter": ("VarianceFilter", NODE_REGISTRY.get_node_config_class("VarianceFilter")()),
    "standard_scaler": ("StandardScaler", NODE_REGISTRY.get_node_config_class("StandardScaler")()),
    "feature_concatenate": ("FeatureConcatenate", NODE_REGISTRY.get_node_config_class("FeatureConcatenate")()),
    "random_forest_regressor": ("RandomForestRegressor", NODE_REGISTRY.get_node_config_class("RandomForestRegressor")()),
    "r2": ("R2Score", NODE_REGISTRY.get_node_config_class("R2Score")()),
    "mse": ("MSE", NODE_REGISTRY.get_node_config_class("MSE")()),
    "sink": ("Sink", NODE_REGISTRY.get_node_config_class("Sink")()),
}

edges = [
    Edge(source="data_source", target="data_missing_filter", ports_map=[("output", "input")]),
    Edge(source="data_missing_filter", target="category_splitter", ports_map=[("output", "input")]),
    Edge(source="one_hot_encoder", target="feature_concatenate", ports_map=[("output", "input_2")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("numerical", "input")]),
    Edge(source="category_splitter", target="iqr_filter", ports_map=[("categorical", "sliced")]),
    Edge(source="target_source", target="iqr_filter", ports_map=[("output", "target")]),
    Edge(source="iqr_filter", target="variance_threshold_filter", ports_map=[("output", "input")]),
    Edge(source="iqr_filter", target="one_hot_encoder", ports_map=[("sliced", "input")]),
    Edge(source="variance_threshold_filter", target="standard_scaler", ports_map=[("output", "input")]),
    Edge(source="standard_scaler", target="feature_concatenate", ports_map=[("output", "input_1")]),
    Edge(source="feature_concatenate", target="random_forest_regressor", ports_map=[("output", "X")]),
    Edge(source="iqr_filter", target="target_missing_filter", ports_map=[("target", "input")]),
    Edge(source="target_missing_filter", target="random_forest_regressor", ports_map=[("output", "y")]),
    Edge(source="random_forest_regressor", target="r2", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="r2", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="mse", ports_map=[("pred", "pred")]),
    Edge(source="target_missing_filter", target="mse", ports_map=[("output", "target")]),
    Edge(source="random_forest_regressor", target="sink", ports_map=[("pred", "dump")]),
]

pipe = Pipeline(config=PipelineConfig(nodes=nodes, edges=edges, name="Tuning Demo Pipeline"))
pipe.compile()
print("Pipeline compiled with", len(pipe.node_objects), "nodes.")

## 2. Instantiate the Tuner

`RayPipelineTuner` takes the compiled pipeline and a `RayPipelineTunerConfig`. The tuner's own config nests a `SmartRunnerConfig`, which controls the runner used inside each trial. Defaults work for a standard train/evaluate cycle.

On construction the tuner:

- **Snapshots** `pipeline.config` so subsequent edits to the original pipeline do not leak into trials.
- Builds an internal "dummy" compiled pipeline that is only used for introspection (no data flows through it).

In [ ]:
from tsut.core.pipeline.tuners.ray_tuner import (
    RayPipelineTuner,
    RayPipelineTunerConfig,
)

tuner = RayPipelineTuner(pipe, config=RayPipelineTunerConfig())
tuner

## 3. Inspect the Default Hyperparameter Space

`default_hyperparameter_space()` walks every node in the pipeline and collects its `hyperparameter_space` (when defined) into a flat dict keyed by `"<node_name>/<hp_name>"`. The values are Ray Tune search-space objects (`tune.choice`, `tune.uniform`, …).

You can use this dict as-is, or edit it before handing it to `tune()`.

In [ ]:
default_space = tuner.default_hyperparameter_space()
for key, search_space in default_space.items():
    print(f"{key:<45} -> {search_space}")

## 4. Customise the Search Space

The returned dict is ordinary Python — you can add, remove, or override entries freely. Below we:

- narrow the `RandomForestRegressor.n_estimators` options,
- widen the `MissingRateFilter.threshold` range for the **data** stream,
- and **drop** `target_missing_filter/threshold` from the space. The pipeline config pins it to `0.0` (drop any row with a missing target); letting Ray resample it would leak NaNs into the regressor. Keys removed from the search space fall back to the base-config value.

In [ ]:
from ray import tune

param_space = tuner.default_hyperparameter_space()
param_space["random_forest_regressor/n_estimators"] = tune.choice([25, 50, 100])
param_space["random_forest_regressor/max_depth"] = tune.choice([None, 5, 10])
param_space["data_missing_filter/threshold"] = tune.uniform(0.1, 0.9)
# Keep the target filter pinned to its base-config 0.0 by removing the key.
param_space.pop("target_missing_filter/threshold", None)

param_space

### Initialising Ray

Ray's `uv` runtime-env hook tries to run each worker through `uv run`, which requires the project's `pyproject.toml` to sit inside the Ray worker's `working_dir`. When running this notebook from `examples/`, we:

1. Disable the hook (`RAY_ENABLE_UV_RUN_RUNTIME_ENV=0`) so workers inherit the driver's Python environment directly.
2. Point Ray's `working_dir` at the repo root so file references in the pipeline (code, data) resolve the same way on workers as on the driver.

If Ray is already initialised (or you're running from the repo root with no `uv` workflow), these settings are harmless no-ops.

In [ ]:
import os

import ray

os.environ.setdefault("RAY_ENABLE_UV_RUN_RUNTIME_ENV", "0")

if not ray.is_initialized():
    ray.init(runtime_env={"working_dir": ".."})

## 5. Run the Tuning Loop

`tune()` accepts the search space plus either:

- `optimization_metric` — the name of a single metric node to optimise, or
- `metric_aggregator` — a callable `dict[str, float] -> float` that reduces every metric node's scalar output to a single objective.

Under the hood each trial:

1. Deep-copies the snapshot config and patches in the sampled hyperparameters.
2. Compiles a fresh `Pipeline` and wraps it in a `SmartRunner`.
3. Calls `runner.train()` then `runner.evaluate()`.
4. Reduces the metrics to a scalar (`optimization_metric` field in the reported metrics) and reports it to Ray Tune.

The snippet below runs a quick search (4 trials, minimising MSE).

In [ ]:
from ray.tune import TuneConfig

tuner.tune(
    param_space=param_space,
    optimization_metric="mse",
    tune_config=TuneConfig(num_samples=4, mode="min"),
)

## 6. Inspect the Best Result

`get_best()` returns the best `ray.tune.Result` object. By default it ranks on the synthetic `optimization_metric` column the tuner injected, but you can pick any reported metric (e.g. `"r2"` with `mode="max"`).

In [ ]:
best = tuner.get_best(mode="min")

print("Best hyperparameters:")
for k, v in best.config.items():
    print(f"  {k}: {v}")

print("\nBest metrics:")
for k in ("optimization_metric", "mse", "r2"):
    if k in best.metrics:
        print(f"  {k}: {best.metrics[k]:.4f}")

You can also rank by any other metric that the pipeline reports. Here we pick the trial that maximised `r2`:

In [ ]:
best_r2 = tuner.get_best(metric="r2", mode="max")
print("Best R2:", best_r2.metrics["r2"])
print("Config:", best_r2.config)

## 7. Using a `metric_aggregator` Instead

When a single metric is not enough, pass a `metric_aggregator` callable. It receives the full dict of metric-node outputs and must return one float. Below we minimise `mse` while penalising low `r2`:

In [ ]:
def composite_objective(metrics: dict[str, float]) -> float:
    """Lower is better. R2 penalty shrinks with better fit."""
    return metrics["mse"] + 10.0 * (1.0 - metrics["r2"])


tuner_composite = RayPipelineTuner(pipe, config=RayPipelineTunerConfig())
tuner_composite.tune(
    param_space=param_space,
    metric_aggregator=composite_objective,
    tune_config=TuneConfig(num_samples=4, mode="min"),
)

best_composite = tuner_composite.get_best(mode="min")
print("Best composite objective:", best_composite.metrics["optimization_metric"])
print("Config:", best_composite.config)

## 8. Passing `input_data` Efficiently

If your pipeline uses `InputsPassthrough`-style sources (instead of loading from disk inside the pipeline), each trial needs the input arrays. The tuner accepts an `input_data` kwarg:

```python
tuner.tune(
    param_space=param_space,
    optimization_metric="mse",
    input_data={
        "source": {
            "X": (X_array, X_context),
            "y": (y_array, y_context),
        }
    },
)
```

Internally the tuner uses `ray.tune.with_parameters` so the payload is pushed to Ray's object store **once** and every trial picks up a reference, rather than re-pickling the arrays per trial.

## Recap

- `RayPipelineTuner` auto-discovers the hyperparameter space from nodes exposing `hyperparameter_space`.
- Every trial rebuilds the pipeline from the snapshot config, so trials are isolated.
- The objective is chosen via `optimization_metric` (single metric) or `metric_aggregator` (custom reducer).
- `get_best()` returns a `ray.tune.Result`; use `metric=` / `mode=` to rank by any reported metric.
- Use `input_data=` to feed in-memory arrays efficiently through Ray's object store.